In [1]:
import json

In [2]:
llms = ["gemma2_9b", "llama3.1_8b", "mistral_7b", "qwen2.5_7b"]
rms = ["fsfairx_rm", "mistral_rm"]

llm_mapping = {
    "gemma2_9b": "google/gemma-2-9b-it", 
    "llama3.1_8b": "meta-llama/Llama-3.1-8B-Instruct",
    "llama3.2_3b": "meta-llama/Llama-3.2-3B-Instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen2.5_7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen3_4b": "Qwen/Qwen3-4B-Instruct-2507",
}

rm_mapping = {
    "fsfairx_rm": "sfairXC/FsfairX-LLaMA3-RM-v0.1", 
    "mistral_rm": "weqweasdas/RM-Mistral-7B"
}

In [3]:
dataset = "alpaca"
rm = "fsfairx_rm"
llm = "mistral_7b"
distribution = "shifted_exponential"
transformation = "cdf"
batch_size = "1"
alpha = "0.99"

data = []
with open(f"../slurm_result_{dataset}/result_3_{rm}_{llm}_{distribution}_{transformation}_bs{batch_size}_a{alpha}.json") as f:
    data = json.load(f)

In [5]:
data.keys()

dict_keys(['llm_name', 'rm_name', 'distribution', 'transformation', 'batch_size', 'epoch', 'alpha', 'delta', 'costs', 'per_prompt', 'aggregate'])

In [7]:
data["aggregate"]

[{'cost': 0.02,
  'win_rate_median': 0.5225,
  'win_rate_p25': 0.505,
  'win_rate_p75': 0.535,
  'avg_sample_count_median': 15.895,
  'avg_sample_count_p25': 14.717500000000001,
  'avg_sample_count_p75': 17.2125},
 {'cost': 0.01,
  'win_rate_median': 0.54,
  'win_rate_p25': 0.52,
  'win_rate_p75': 0.55,
  'avg_sample_count_median': 23.27,
  'avg_sample_count_p25': 20.1475,
  'avg_sample_count_p75': 26.572499999999998},
 {'cost': 0.008,
  'win_rate_median': 0.5425,
  'win_rate_p25': 0.53,
  'win_rate_p75': 0.565,
  'avg_sample_count_median': 26.42,
  'avg_sample_count_p25': 22.447499999999998,
  'avg_sample_count_p75': 30.6525},
 {'cost': 0.006,
  'win_rate_median': 0.55,
  'win_rate_p25': 0.535,
  'win_rate_p75': 0.565,
  'avg_sample_count_median': 30.71,
  'avg_sample_count_p25': 25.7525,
  'avg_sample_count_p75': 35.8825},
 {'cost': 0.004,
  'win_rate_median': 0.56,
  'win_rate_p25': 0.54,
  'win_rate_p75': 0.5762499999999999,
  'avg_sample_count_median': 38.4,
  'avg_sample_count_p2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

def plot_win_rate_analysis(ax2, data_point_dict):
    """
    Comprehensive plotting function for Pandora's Box vs Best of N win rate analysis.
    
    Parameters:
    -----------
    data : list
        Input data containing NAs and algorithm comparisons
    """

    keys = list(data_point_dict.keys())
    key_labels = [llm_mapping[key] for key in keys]
    data_point_list = [data_point_dict[k] for k in keys]
    xs_all = [[item[0] for item in data_point[0]] for data_point in data_point_list]
    ys_all = [[item[1] for item in data_point[0]] for data_point in data_point_list]
    
    # Create figure with subplots
    # fig, axes = plt.subplots(1, 1, figsize=(8, 6))
    # fig.suptitle('Pandora\'s Box vs Best of N: Win Rate Analysis', fontsize=16, fontweight='bold')
    
    # ========== Plot 2: Binned statistics ==========
    # ax2 = axes[0, 0]
    # ax2 = axes    
    # Discretize into bins of size 50
    bin_size = 50
    min_sample = min([min(xs) for xs in xs_all])
    max_sample = max([max(xs) for xs in xs_all])
    bins = np.arange(min_sample, max_sample + bin_size, bin_size)

    bin_stats_all = []
    bin_centers_all = []
    for j in range(len(xs_all)):
        xs = xs_all[j]
        ys = ys_all[j]

        # Calculate statistics for each bin
        bin_stats = []
        bin_centers = []
        
        for i in range(len(bins) - 1):
            bin_mask = [(bins[i] <= x < bins[i + 1]) for x in xs]
            bin_values = [y for y, mask in zip(ys, bin_mask) if mask]
            
            if bin_values:
                bin_center = (bins[i] + bins[i + 1]) / 2
                bin_centers.append(bin_center)
                bin_stats.append({
                    'mean': np.mean(bin_values),
                    'median': np.median(bin_values),
                    'q25': np.percentile(bin_values, 25),
                    'q75': np.percentile(bin_values, 75),
                    'std': np.std(bin_values),
                    'count': len(bin_values)
                })
        bin_stats_all.append(bin_stats)
        bin_centers_all.append(bin_centers)
    
    # Extract statistics
    # means = [s['mean'] for s in bin_stats]
    medians = [[s['median'] for s in bin_stats] for bin_stats in bin_stats_all]
    # q25s = [s['q25'] for s in bin_stats]
    # q75s = [s['q75'] for s in bin_stats]
    
    # Plot statistics
    # ax2.plot(bin_centers, means, 'o-', label='Mean', linewidth=2, markersize=8)
    for i in range(len(bin_stats_all)):
        ax2.plot(bin_centers_all[i], medians[i], 's-', label=f'{key_labels[i]}', linewidth=1.5, markersize=6)
    # ax2.fill_between(bin_centers, q25s, q75s, alpha=0.3, label='25-75 percentile')
    ax2.set_xlabel('Sample Count (binned)', fontsize=12)
    ax2.set_ylabel('Win Rate Statistics', fontsize=12)
    ax2.set_title(f'Win Rates by Sample Count', fontsize=13)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # plt.tight_layout()
    # plt.show()
    
    return ax2, bin_stats


# out = plot_win_rate_analysis({"test": data_point})